In [ ]:
#prep data for fine-tuning 1


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc

# Keep Scanpy plots compact for notebook viewing
sc.settings.set_figure_params(dpi=100, dpi_save=100)

PROJECT_DIR = Path.cwd()
DATA_FILE = PROJECT_DIR / "data" / "nsclc" / "nsclc_integrated.h5ad"

INPUT_DIR = PROJECT_DIR / "input_data" / "01_primary_celltype"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_FILE)

In [ ]:
#load backened
adata_b = sc.read_h5ad(DATA_FILE, backed="r")

print(adata_b.shape)
print(adata_b.obs.columns.tolist())

In [ ]:
#selected cells
KEEP_CELL_TYPES = [
    "Macrophage alveolar",
    "Macrophage",
    "Monocyte",
    "T cell CD4",
    "T cell CD8",
    "NK cell",
    "B cell",
    "Endothelial cell",
    "cDC2",
]

obs = adata_b.obs.copy()

mask = (
    obs["cell_type_major"].isin(KEEP_CELL_TYPES)
    & obs["donor_id"].notna()
    & obs["total_counts"].notna()
)

if "doublet_status" in obs.columns:
    mask = mask & (obs["doublet_status"].astype(str).str.lower() == "singlet")

obs_sel = obs.loc[mask].copy()

print(obs_sel.shape)
print(obs_sel["cell_type_major"].value_counts())

In [ ]:
# Cell 4 — smaller donor-aware sampling
# per donor 1000 crashes Kernel
MAX_PER_CLASS = 3000
MAX_PER_DONOR_PER_CLASS = 300
RANDOM_STATE = 42

selected_cells = []

for cell_type, df_ct in obs_sel.groupby("cell_type_major"):
    capped_parts = []

    for donor, df_donor in df_ct.groupby("donor_id"):
        sampled_donor = df_donor.sample(
            n=min(MAX_PER_DONOR_PER_CLASS, len(df_donor)),
            random_state=RANDOM_STATE
        )
        capped_parts.append(sampled_donor)

    df_ct_capped = pd.concat(capped_parts)

    df_ct_final = df_ct_capped.sample(
        n=min(MAX_PER_CLASS, len(df_ct_capped)),
        random_state=RANDOM_STATE
    )

    selected_cells.extend(df_ct_final.index.tolist())

selected_cells = pd.Index(selected_cells)
selected_obs = obs_sel.loc[selected_cells].copy()

print("Selected cells:", len(selected_cells))
print(selected_obs["cell_type_major"].value_counts())

print("\nTop donors:")
print(
    selected_obs
    .groupby(["disease", "donor_id"])
    .size()
    .reset_index(name="cell_count")
    .sort_values("cell_count", ascending=False)
    .head(20)
)

In [ ]:
#donor check
donor_summary = (
    selected_obs
    .groupby(["donor_id", "disease"])
    .size()
    .reset_index(name="cell_count")
    .sort_values("cell_count", ascending=False)
)

total_selected = len(selected_obs)

donor_summary["fraction_of_selected"] = donor_summary["cell_count"] / total_selected

display(donor_summary.head(20))

print("Max donor fraction:")
print(donor_summary["fraction_of_selected"].max())

In [ ]:
# Cell 5 — load only selected cells into memory safely

print("Cells selected:", len(selected_cells))
print("Genes:", adata_b.n_vars)

# Load selected cells only
adata = adata_b[selected_cells, :].to_memory()

# Close backed object to release file handle
adata_b.file.close()

print("Loaded into memory:")
print(adata)

print("\nLabel counts:")
print(adata.obs["cell_type_major"].value_counts())

print("\nTop donors in loaded object:")
print(
    adata.obs
    .groupby(["disease", "donor_id"])
    .size()
    .reset_index(name="cell_count")
    .sort_values("cell_count", ascending=False)
    .head(20)
)

# quick sanity check
assert adata.n_obs == len(selected_cells)
assert adata.n_vars == 17764

In [ ]:
# Required Geneformer metadata
adata.obs["celltype"] = adata.obs["cell_type_major"].astype(str)
adata.obs["disease"] = adata.obs["disease"].astype(str)
adata.obs["individual"] = adata.obs["donor_id"].astype(str)
adata.obs["cell_id"] = adata.obs_names.astype(str)
adata.obs["filter_pass"] = 1
adata.obs["n_counts"] = adata.obs["total_counts"].astype(float)

adata.var["ensembl_id"] = adata.var.index.astype(str)

adata.obsm = {}
adata.uns = {}
adata.varm = {}
adata.obsp = {}

adata.obs = adata.obs[
    ["cell_id", "individual", "celltype", "disease", "n_counts", "filter_pass"]
].copy()

print(adata)
print(adata.obs["celltype"].value_counts())
print(adata.obs["disease"].value_counts())

In [ ]:
# Cell 7 — save Geneformer-ready h5ad

outfile = INPUT_DIR / "01_primary_celltype.h5ad"

adata.write_h5ad(outfile)

print("Saved:")
print(outfile)

print("\nFile size:")
print(f"{outfile.stat().st_size / 1e9:.2f} GB")

In [ ]:
# Cell 8 — quick verification

adata_check = sc.read_h5ad(outfile, backed="r")

print(adata_check)
print(adata_check.obs.head())
print(adata_check.var.head())

adata_check.file.close()